# Text Classification with Lexos: Federalist Papers Example

This notebook demonstrates a complete workflow for **text classification** using the `lexos` library. It walks through:

- Loading and preprocessing a collection of `.txt` files
- Cleaning and tokenizing the text
- Creating a Document-Term Matrix (DTM)
- Training a machine learning classifier
- Predicting categories for documents
- Saving and exporting classification results

---

## Dataset: The Federalist Papers

The example data used here comes from **The Federalist Papers**, a series of 85 essays written in the late 18th century to promote the ratification of the U.S. Constitution.

These essays were authored by **Alexander Hamilton**, **James Madison**, and **John Jay**, and are often used in authorship attribution studies because:

- Some essays have disputed authorship, making them useful for classification tasks

The `.txt` files are organized in folders named after the authors (e.g., `Hamilton`, `Madison`, `Jay`), which are used as **class labels** during training.




In [1]:
from pathlib import Path
from lexos.io.loader import Loader
from lexos.scrubber.scrubber import Scrubber
from lexos.tokenizer import Tokenizer
from lexos.dtm import DTM, Vectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

## Preparing the Documents

This code demonstrates how to prepare text documents for classification by cleaning, tokenizing, and building a Document-Term Matrix (DTM). Below is a step-by-step explanation:

In [ ]:
from pathlib import Path
from lexos.scrubber.scrubber import Scrubber
from lexos.tokenizer import Tokenizer
from lexos.dtm import DTM
from lexos.dtm import DTM, Vectorizer

- **Path** helps handle file system paths.

- **Scrubber** is used to clean text (e.g., removing digits, punctuation, etc.).

- **Tokenizer** converts cleaned text into tokens using a language model (`en_core_web_sm`).

- **DTM** and **Vectorizer** are used to create a Document-Term Matrix from tokens.


## Loading Text Files and Debugging

In [ ]:
# Step 1: Choose or verify the folder path
# Recommend placing .txt files in a known subfolder like "sample_files"
FILES_DIR = Path("sample_files")  # Relative path easy to adapt

print(f"Looking for .txt files in: {FILES_DIR.resolve()}")

#Step 2: Check if folder exists
if not FILES_DIR.exists():
    print("ERROR: The directory does not exist. Please check the folder name or path.")
else:
    #Step 3: Search for .txt files recursively
    files = list(FILES_DIR.rglob("*.txt"))
    
    if not files:
        print("WARNING: No .txt files found in the directory or subdirectories.")
    else:
        print(f"Found {len(files)} .txt file(s).")
        print("Example file:", files[0])


## Initializing the Text Preprocessing Pipeline

In [ ]:
scrubber = Scrubber()
scrubber.add_pipe("lower_case")
scrubber.add_pipe("digits")
scrubber.add_pipe("punctuation")

tokenizer = Tokenizer(model="en_core_web_sm")

Creates a **Scrubber** object and adds cleaning steps:

- Convert all text to lowercase.
- Remove digits.
- Remove punctuation.

Initializes a **Tokenizer** using the English spaCy model to generate tokens.


##  Preparing Containers for Data

In [ ]:
# store tokens for each document
token_lists = []
# stores the file names to use as row identifiers in the document-term matrix (DTM)
row_labels = []
# stores the true class/category label for each document 
target_labels = []
docs = []  # to store spacy docs

- **token_lists**: will store lists of tokens for each document.

- **row_labels**: will store file names as identifiers (for matrix rows).

- **target_labels**: will store the true class/category for each document (based on the parent folder name).

- **docs**: will store spaCy Doc objects for each document.


## Processing Each Text File

In [ ]:
# loop through each text file found in the list of files
for f in files:
    raw = f.read_text(encoding="utf-8")
    #clean the text using the scrubber(lowercasing, removing digits/punctuation)
    clean = scrubber.scrub(raw)
    #convert the cleaned text into a spaCy Doc object using tokenizer
    doc = tokenizer.make_doc(clean)
    # extract tokens/words from the Doc and store as a list of strings
    tokens = [token.text for token in doc]
    #add the list of tokens for each file to the main collection 
    token_lists.append(tokens)
    docs.append(doc) 
    #store the file name as the row label for the DTM (helps track which row is which document)
    row_labels.append(f.name)          
    # store the name of the parent folder (in this case author madison, jay or hamilton) as the target label
    target_labels.append(f.parent.name)  

For every text file:

- Read raw text content.

- Clean the text using the scrubber pipeline.

- Tokenize the cleaned text into a spaCy `Doc`.

- Extract token texts as a list of strings.

- Append tokens to `token_lists`.

- Save the `Doc` object.

- Use the file name as the document identifier (`row_labels`).

- Use the parent folder name as the document's true label (`target_labels`).  
  *(Assuming folder names correspond to class labels, e.g., author names)*


##  Building the Document-Term Matrix (DTM)

In [ ]:
#build the document term matrix
dtm = DTM(vectorizer=Vectorizer())
feature_matrix = dtm(docs=token_lists, labels=row_labels)
feature_matrix = dtm.doc_term_matrix
 


Found 80 files


- Initializes a **DTM** object with a **Vectorizer**.

- Passes the list of token lists and document labels to build the feature matrix.

- Extracts the sparse Document-Term Matrix (`doc_term_matrix`), which represents word counts for each document.


## Classification and Prediction Explanation

This section handles training, predicting, and saving results using functions from `lexos.classification`.

- **train_classifier**: Trains a classifier using feature matrix and labels.

- **predict_labels**: Uses the trained classifier to predict labels for new data.

- **PredictionSaver**: Utility class for saving predictions to CSV or as record objects.


In [ ]:
# import sys
from pathlib import Path

# #import
from lexos.classification import train_classifier, predict_labels, PredictionSaver


Classification Report
              precision    recall  f1-score   support

    Hamilton       0.90      0.95      0.92        19
         Jay       1.00      0.67      0.80         3
     Madison       0.80      0.80      0.80        10

    accuracy                           0.88        32
   macro avg       0.90      0.80      0.84        32
weighted avg       0.88      0.88      0.87        32

Predictions saved to predictions.csv
80 records created with classification metadata.


## Train the Classifier

In [ ]:
clf, report = train_classifier(feature_matrix, target_labels)
print("\nClassification Report")
print(report)

- Trains a model (`clf`) on the provided `feature_matrix` and `target_labels`.

- Outputs a classification report showing metrics like precision, recall, and F1-score.

## Predict Labels

In [ ]:
# Predict (on same data here)
new_feature_matrix = feature_matrix
predicted_labels = predict_labels(clf, new_feature_matrix)

- Uses the trained classifier to predict labels.

- In this example, it's predicting on the same data it was trained on (for demo purposes).

## Save Predictions to CSV

In [ ]:
# Save predictions
PredictionSaver.save_to_csv(row_labels, predicted_labels, "predictions.csv")

- Saves the predicted labels with their corresponding document names to a CSV file.

- Useful for reviewing results or further analysis.


## Save Predictions as Record Objects

In [ ]:
# Record objects
records = PredictionSaver.save_to_records(docs, row_labels, predicted_labels)

- Converts the predictions into `Record` objects for downstream processing or visualization.

- Includes the original `Doc`, label, and metadata.


### **Explanation of the Classification Report**

- **precision**:  
  The proportion of predicted labels that were actually correct for each class.  
  Example: `Hamilton` has precision 1.00 → every essay the model labeled as Hamilton *was* truly written by Hamilton.

- **recall**:  
  The proportion of true labels that were correctly found by the model.  
  Example: `Hamilton` has recall 0.95 → the model found 95% of Hamilton’s essays.

- **f1-score**:  
  The precision and recall, how balanced it was overall
  Example: `Hamilton` has f1-score 0.98 → overall very high balanced performance.

- **support**:  
  The number of actual documents for each author in the test set.  
  Example: there were 21 Hamilton essays, 2 Jay essays, 9 Madison essays.

---

- **accuracy**:  
  The overall percentage of correct predictions across all documents.  
  Example: 0.97 → means 97% of the test documents were classified correctly.

- **macro avg**:  
  Average of precision, recall, and f1-score treating each class equally (unweighted).  
  Good for seeing if performance is balanced across all authors.

- **weighted avg**:  
  Average of precision, recall, and f1-score weighted by each class’s support.  
  So more common authors (like Hamilton) influence this more.


